# End-to-End Data Science Project — Customer Churn

This notebook demonstrates an end-to-end workflow using synthetic data: data validation, EDA, train/test splitting, baseline comparison, preprocessing/model pipeline, evaluation, artifact saving, and a basic prediction interface. It does not contact external services or create cloud resources.

## 1. Imports and reproducibility


In [ ]:
import os, json, platform
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report)

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
print('Python:', platform.python_version())
print('pandas:', pd.__version__, '| scikit-learn:', sklearn.__version__)
print('Random state:', RANDOM_STATE)

## 2. Define the business problem

**Decision:** help a retention team prioritize review of active accounts that may cancel in the next 30 days.

**Target:** `churn` (1 = churn, 0 = retained) in this synthetic demonstration.

**Caution:** a prediction is not proof that an account will churn, and model performance on synthetic data does not establish real-world usefulness.

## 3. Generate synthetic data


In [ ]:
n = 1400
tenure = rng.integers(1, 73, size=n)
monthly_fee = np.clip(rng.normal(72, 19, size=n), 18, 160).round(2)
support_tickets = rng.poisson(1.3, size=n)
contract_type = rng.choice(['monthly', 'annual', 'two_year'], size=n, p=[0.56, 0.29, 0.15])
payment_method = rng.choice(['card', 'bank_transfer', 'digital_wallet'], size=n, p=[0.50, 0.30, 0.20])
logit = (0.8 - 0.04 * tenure + 0.022 * (monthly_fee - 70)
         + 0.30 * support_tickets
         + np.where(contract_type == 'monthly', 0.8, 0)
         + np.where(payment_method == 'digital_wallet', 0.15, 0))
p = 1 / (1 + np.exp(-logit))
churn = rng.binomial(1, p)
df = pd.DataFrame({
    'tenure_months': tenure,
    'monthly_fee': monthly_fee,
    'support_tickets': support_tickets,
    'contract_type': contract_type,
    'payment_method': payment_method,
    'churn': churn,
})
missing_rows = rng.choice(df.index, size=40, replace=False)
df.loc[missing_rows, 'monthly_fee'] = np.nan
df.head()

## 4. Validate the input schema and audit quality


In [ ]:
REQUIRED_FEATURES = ['tenure_months', 'monthly_fee', 'support_tickets', 'contract_type', 'payment_method']
TARGET = 'churn'

def validate_required_columns(frame, required):
    missing = sorted(set(required) - set(frame.columns))
    if missing:
        raise ValueError(f'Missing required columns: {missing}')
    return True

validate_required_columns(df, REQUIRED_FEATURES + [TARGET])
print('Shape:', df.shape)
print('Duplicate rows:', int(df.duplicated().sum()))
display(df.isna().sum().rename('missing_count').to_frame())
display(df[TARGET].value_counts().rename('count').to_frame())
print('Target prevalence:', round(df[TARGET].mean(), 3))

## 5. Explore the data

Use these plots to describe the generated sample only. Group sizes matter when comparing rates.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
df['tenure_months'].hist(bins=15, ax=axes[0])
axes[0].set_title('Tenure distribution')
axes[0].set_xlabel('Months')
df['monthly_fee'].hist(bins=15, ax=axes[1])
axes[1].set_title('Monthly fee distribution')
axes[1].set_xlabel('Fee (synthetic units)')
contract_summary = df.groupby('contract_type')['churn'].agg(['size', 'mean']).sort_values('mean')
contract_summary['mean'].plot(kind='bar', ax=axes[2])
axes[2].set_title('Churn rate by contract')
axes[2].set_ylabel('Observed rate')
axes[2].tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()
display(contract_summary.rename(columns={'size':'customers','mean':'churn_rate'}))

## 6. Split before fitting preprocessing

In a real application, validate that every feature is available at the prediction timestamp. Use time-based or group-aware splitting if the data structure requires it.

In [ ]:
X = df[REQUIRED_FEATURES].copy()
y = df[TARGET].copy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)
numeric_features = ['tenure_months', 'monthly_fee', 'support_tickets']
categorical_features = ['contract_type', 'payment_method']
print('Train rows:', len(X_train), '| Test rows:', len(X_test))
print('Train target rate:', round(y_train.mean(), 3), '| Test target rate:', round(y_test.mean(), 3))

## 7. Fit a baseline


In [ ]:
baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)
print('Baseline accuracy:', round(accuracy_score(y_test, baseline_pred), 3))
print('Baseline positive-class recall:', round(recall_score(y_test, baseline_pred, zero_division=0), 3))

## 8. Build and train the full preprocessing/model pipeline


In [ ]:
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore')),
])
preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, numeric_features),
    ('categorical', categorical_pipeline, categorical_features),
])
model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced')),
])
model.fit(X_train, y_train)
pred = model.predict(X_test)
probabilities = model.predict_proba(X_test)[:, 1]
metrics = {
    'accuracy': accuracy_score(y_test, pred),
    'precision': precision_score(y_test, pred, zero_division=0),
    'recall': recall_score(y_test, pred, zero_division=0),
    'f1': f1_score(y_test, pred, zero_division=0),
    'roc_auc': roc_auc_score(y_test, probabilities),
}
pd.Series(metrics, name='score').round(3).to_frame()

## 9. Evaluate errors and compare to baseline


In [ ]:
print('Confusion matrix (actual rows, predicted columns):')
print(confusion_matrix(y_test, pred, labels=[0, 1]))
print(classification_report(y_test, pred, zero_division=0))
comparison = pd.DataFrame([
    {'model': 'most-frequent baseline', 'accuracy': accuracy_score(y_test, baseline_pred),
     'precision': precision_score(y_test, baseline_pred, zero_division=0),
     'recall': recall_score(y_test, baseline_pred, zero_division=0)},
    {'model': 'logistic regression pipeline', 'accuracy': metrics['accuracy'],
     'precision': metrics['precision'], 'recall': metrics['recall']},
])
comparison.round(3)

## 10. Demonstrate a threshold policy

This is illustrative only. In a real project, choose a threshold using validation data and documented costs/capacity, then evaluate once on the final test set.

In [ ]:
threshold = 0.60
threshold_pred = (probabilities >= threshold).astype(int)
print('Threshold:', threshold)
print('Precision:', round(precision_score(y_test, threshold_pred, zero_division=0), 3))
print('Recall:', round(recall_score(y_test, threshold_pred, zero_division=0), 3))
print('Cases flagged:', int(threshold_pred.sum()), '/', len(threshold_pred))

## 11. Package the trained pipeline (optional local artifact)

The following writes a local file. Only load serialized artifacts from trusted sources; joblib/pickle artifacts can execute code during deserialization.

In [ ]:
try:
    import joblib
    artifact_path = Path('churn_pipeline.joblib')
    joblib.dump(model, artifact_path)
    print('Saved trusted local artifact:', artifact_path.resolve())
except ImportError:
    print('Install joblib to save the model artifact: pip install joblib')

## 12. Create a reusable prediction function and validate its input


In [ ]:
def predict_churn(frame, fitted_model, required_features=REQUIRED_FEATURES):
    missing = sorted(set(required_features) - set(frame.columns))
    if missing:
        raise ValueError(f'Missing required features: {missing}')
    ordered = frame[required_features].copy()
    prediction = fitted_model.predict(ordered)
    result = pd.DataFrame({'prediction': prediction}, index=frame.index)
    if hasattr(fitted_model, 'predict_proba'):
        result['churn_probability'] = fitted_model.predict_proba(ordered)[:, 1]
    return result

sample = X_test.head(5)
display(predict_churn(sample, model))
try:
    predict_churn(sample.drop(columns=['contract_type']), model)
except ValueError as exc:
    print('Expected validation error:', exc)

## 13. Draft an operational monitoring plan

Complete this table for your own project before deployment. The alert thresholds must come from baselines, load tests, and business requirements.

In [ ]:
monitoring_plan = pd.DataFrame([
    {'signal': 'Schema / missingness', 'purpose': 'Catch input contract or upstream data changes', 'response': 'Pause scoring and investigate source'},
    {'signal': 'Request error rate', 'purpose': 'Detect serving failures', 'response': 'Inspect deployment, logs, and dependencies'},
    {'signal': 'p95 latency', 'purpose': 'Check service responsiveness', 'response': 'Inspect load, model runtime, and resource limits'},
    {'signal': 'Prediction distribution', 'purpose': 'Spot unusual score shifts', 'response': 'Check data drift and model version'},
    {'signal': 'Delayed outcome metrics', 'purpose': 'Assess real performance when labels arrive', 'response': 'Review model and data; consider retraining'},
])
monitoring_plan

## 14. Final project summary template

Before presenting, fill in:

- **Problem and decision:** what action will this support?
- **Data:** source/generator, schema, target definition, and limitations.
- **Method:** baseline, pipeline, validation strategy, and model version.
- **Evaluation:** metrics, error slices, and threshold rationale.
- **Operations:** serving approach, monitoring, rollback, privacy, and cost.
- **Recommendation:** pilot, no-go, or more evidence needed—and why.

## Further extensions

1. Add cross-validation or time-aware evaluation.
2. Add automated tests for schema and prediction behavior.
3. Build a FastAPI endpoint with Pydantic request validation.
4. Create a Dockerfile and CI workflow.
5. Replace the synthetic data with an approved public dataset and document provenance.